# RecoverAI UPI dataset profile

This notebook profiles the raw Kaggle UPI transaction data before synthetic customer assignment, recovery-label simulation, or model training. It treats the observed CSV headers as the input contract and does not infer recovery outcomes.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        raw_path = candidate / "ml" / "data" / "raw" / "upi_transactions_2024.csv"
        if raw_path.exists():
            return candidate
    raise FileNotFoundError("Could not locate ml/data/raw/upi_transactions_2024.csv")


ROOT = find_repo_root()
RAW_PATH = ROOT / "ml" / "data" / "raw" / "upi_transactions_2024.csv"
df = pd.read_csv(RAW_PATH)

print(f"Shape: {df.shape}")
df.head()

In [ ]:
print("Exact headers:")
print(df.columns.tolist())

print("\nDtypes:")
print(df.dtypes)

print("\nDataFrame info:")
df.info()

In [ ]:
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nUnique values:")
print(df.nunique().sort_values())

In [ ]:
numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "category", "string"]).columns.tolist()

print("Numeric:")
print(numeric_cols)
print("\nCategorical:")
print(categorical_cols)

In [ ]:
status_counts = df["transaction_status"].value_counts()
status_percent = df["transaction_status"].value_counts(normalize=True).mul(100).round(2)

print("Status counts:")
print(status_counts)
print("\nStatus percent:")
print(status_percent)

print("\nOverall amount profile:")
print(df["amount (INR)"].describe())
print("\nAmount profile by status:")
print(df.groupby("transaction_status")["amount (INR)"].describe())

In [ ]:
# Exclude identifiers and timestamps: their high cardinality makes value-count tables uninformative.
profile_categoricals = [
    column
    for column in categorical_cols
    if column not in {"transaction id", "timestamp"}
]

for column in profile_categoricals:
    print(f"\n{'=' * 60}\n{column}")
    print(df[column].value_counts(dropna=False).head(20))

In [ ]:
for column in profile_categoricals:
    if column == "transaction_status":
        continue

    failure_rate = (
        df.groupby(column, observed=True)["transaction_status"]
        .apply(lambda values: values.eq("FAILED").mean() * 100)
        .sort_values(ascending=False)
    )
    print(f"\nFailure rate (%) by {column}")
    print(failure_rate.head(20).round(3))

In [ ]:
timestamps = pd.to_datetime(df["timestamp"], errors="coerce")
assert timestamps.notna().all(), "Invalid timestamps found"

expected_hour = timestamps.dt.hour
expected_day_name = timestamps.dt.day_name()
expected_weekend = timestamps.dt.dayofweek.ge(5).astype("int8")

print("Timestamp range:", timestamps.min(), "to", timestamps.max())
print("Hour mismatches:", int(expected_hour.ne(df["hour_of_day"]).sum()))
print("Day-name mismatches:", int(expected_day_name.ne(df["day_of_week"]).sum()))
print("Weekend mismatches:", int(expected_weekend.ne(df["is_weekend"]).sum()))

hour_failure = (
    df.assign(timestamp=timestamps, hour=expected_hour)
    .groupby("hour")["transaction_status"]
    .apply(lambda values: values.eq("FAILED").mean() * 100)
)
hour_failure.round(3)

In [ ]:
print("Fraud flag counts:")
print(df["fraud_flag"].value_counts().sort_index())

print("\nTransaction status within each fraud flag (%):")
fraud_status = pd.crosstab(
    df["fraud_flag"],
    df["transaction_status"],
    normalize="index",
).mul(100)
fraud_status.round(3)

## Phase 1 interpretation

- `transaction_status` describes the original payment result; it is **not** a recovery label.
- `fraud_flag` should later be evaluated as a policy constraint, not blindly treated as a recoverability feature.
- No persistent customer identifier exists, so customer histories must be generated in a separate, documented phase.
- The raw file already supplies hour, day name, and weekend fields. The deterministic cleaner validates and recomputes these fields from `timestamp` rather than adding redundant columns.